### Links and Resources
| Method / Command | Description |
|------------------|-------------|
| [OPTIMIZE (SQL)](https://docs.databricks.com/en/sql/language-manual/delta-optimize.html) | Compacts small Parquet files, applies Z-ORDER, and applies pending deletion vectors |
| [VACUUM (SQL)](https://docs.databricks.com/en/sql/language-manual/delta-vacuum.html) | Permanently removes old Parquet files that are no longer referenced by the transaction log |
| [DeltaTable.vacuum()](https://docs.delta.io/latest/api/python/spark/index.html) | Python API equivalent of SQL `VACUUM` |
| [Deletion Vectors](https://docs.databricks.com/en/delta/deletion-vectors.html) | Soft-delete mechanism: marks deleted rows in a bitmap without rewriting Parquet files |
| [Auto Optimization](https://docs.databricks.com/en/delta/optimize.html#auto-optimization) | Databricks automatically runs OPTIMIZE after writes when enabled |
| [Z-ORDER BY](https://docs.databricks.com/en/delta/data-skipping.html) | Co-locates related data in the same files to improve read performance via data skipping |

In [ ]:
import os
if not os.getenv("DATABRICKS_RUNTIME_VERSION"):
    from spark_setup import spark, display  # local run through Databricks Connect

# Delta Table Optimization: VACUUM, Deletion Vectors & OPTIMIZE

When you run a `DELETE` on a Delta table, the rows are not immediately wiped from disk. Delta Lake uses one of two strategies to record the deletion:

| Strategy | How it works | When used |
|----------|-------------|----------|
| **Deletion Vectors** | Marks deleted rows in a small bitmap file — original Parquet file is untouched | When `delta.enableDeletionVectors = true` (Databricks default) |
| **File rewrite** | Rewrites the affected Parquet file(s) without the deleted rows | When deletion vectors are disabled |

This deferred approach gives you:
- **Time travel** — earlier versions are still readable because old files are kept.
- **Fast deletes** — deletion vectors avoid an expensive file rewrite.
- **ACID safety** — readers see a consistent snapshot during the operation.

The flip side is that deleted data continues to occupy storage. Three operations manage the lifecycle of these files:

| Operation | Purpose |
|-----------|---------|
| **OPTIMIZE** | Compacts many small Parquet files into fewer large ones; resolves pending deletion vectors into clean files |
| **VACUUM** | Permanently deletes Parquet files that are no longer referenced by the transaction log |
| **Deletion Vectors** | Defers the physical rewrite of deleted data — traded off against storage until OPTIMIZE runs |

## Setup: Create the Demo Table

The same `product_inventory` table from the previous notebook is used here.  
It is recreated fresh so the version numbers in `history()` are predictable.

In [ ]:
from pyspark.sql import Row
from delta.tables import DeltaTable

TABLE = "population_metrics.default.product_inventory"

rows = [
    Row(product_id=1,  name="Laptop Pro 15",      category="Electronics", price=1299.99, in_stock=True),
    Row(product_id=2,  name="Wireless Mouse",      category="Electronics", price=25.49,  in_stock=True),
    Row(product_id=3,  name="USB-C Hub",           category="Electronics", price=39.99,  in_stock=False),
    Row(product_id=4,  name="Standing Desk",       category="Furniture",   price=499.00, in_stock=True),
    Row(product_id=5,  name="Ergonomic Chair",     category="Furniture",   price=349.00, in_stock=True),
    Row(product_id=6,  name="Monitor Arm",         category="Furniture",   price=89.95,  in_stock=False),
    Row(product_id=7,  name="Notebook (A5)",       category="Stationery",  price=4.99,   in_stock=True),
    Row(product_id=8,  name="Ballpoint Pen 10pk",  category="Stationery",  price=2.49,   in_stock=True),
    Row(product_id=9,  name="Sticky Notes 5pk",    category="Stationery",  price=3.79,   in_stock=True),
    Row(product_id=10, name="Whiteboard Marker",   category="Stationery",  price=5.99,   in_stock=False),
    Row(product_id=11, name="Coffee Mug",          category="Kitchen",     price=12.00,  in_stock=True),
    Row(product_id=12, name="Desk Lamp LED",       category="Electronics", price=45.00,  in_stock=True),
]

spark.createDataFrame(rows).write.format("delta").mode("overwrite").saveAsTable(TABLE)
print(f"'{TABLE}' created with 12 rows.")
display(spark.table(TABLE).orderBy("product_id"))

# Part 1 — Deletion Vectors

A **Deletion Vector (DV)** is a small bitmap file stored alongside the Parquet data files.  
Instead of rewriting a Parquet file to remove rows, Delta Lake records *which row offsets* inside the file are deleted in this bitmap.

```
Before DV:  parquet file  →  [row0, row1, row2, row3, row4]   (all valid)
After DV:   parquet file  →  [row0, row1, row2, row3, row4]   (untouched)
            deletion vector  →  {row2 deleted}                (bitmap)
```

When Spark reads the table it applies the bitmap on-the-fly: `row2` is skipped in the result.  
The Parquet file itself is never modified.

### Benefits of Deletion Vectors

| Without Deletion Vectors | With Deletion Vectors |
|--------------------------|-----------------------|
| Delete rewrites the whole Parquet file | Delete writes only a tiny bitmap (~bytes to KB) |
| High I/O cost even for single-row deletes | Near-zero I/O cost for the delete itself |
| No additional storage overhead | Small extra storage for bitmap files |
| Clean files after delete | Files accumulate deleted rows until OPTIMIZE |

### Enabling Deletion Vectors

On Databricks, deletion vectors are enabled by default for new managed tables.  
You can see this in `detail()` under `properties`:
```
delta.enableDeletionVectors -> true
```
You can also set it explicitly:
```sql
ALTER TABLE my_table SET TBLPROPERTIES ('delta.enableDeletionVectors' = 'true');
```

### Inspecting Deletion Vector Status via `detail()`

`detail()` returns a `statistics` column that tracks how many rows and files currently have pending deletion vectors.

| Statistic | Meaning |
|-----------|---------|
| `numDeletionVectors` | Number of Parquet files that have an associated deletion vector |
| `numRowsDeletedByDeletionVectors` | Total number of rows marked as deleted across all DVs |

In [ ]:
dt = DeltaTable.forName(spark, TABLE)

# Check DV status before any deletes
detail = dt.detail().select("properties", "statistics").collect()[0]
print("Table properties:", detail["properties"])
print("Statistics before delete:", detail["statistics"])

### Delete a Row and Observe the Deletion Vector

After deleting one row, re-check `statistics`.  
With deletion vectors enabled you will see `numDeletionVectors = 1` and `numRowsDeletedByDeletionVectors = 1` — the Parquet file was not rewritten.

In [ ]:
# Delete a single row — deletion vector should absorb this
dt.delete("product_id = 3")

dt = DeltaTable.forName(spark, TABLE)   # refresh handle
detail = dt.detail().select("numFiles", "sizeInBytes", "statistics").collect()[0]
print("After deleting product_id = 3:")
print(f"  numFiles  : {detail['numFiles']}")
print(f"  sizeInBytes: {detail['sizeInBytes']}")
print(f"  statistics: {detail['statistics']}")
# numFiles and sizeInBytes remain the same — no Parquet rewrite happened
# statistics shows numRowsDeletedByDeletionVectors = 1, numDeletionVectors = 1

### Deletion Vectors in the Transaction Log

The `history()` operation column shows `DELETE` and the `operationParameters` records the predicate.  
The log entry also contains the path to the deletion vector `.dv` file — though `history()` does not expose it directly, the `_delta_log/` JSON files contain it.

In [ ]:
display(
    dt.history()
    .select("version", "timestamp", "operation", "operationParameters", "operationMetrics")
    .orderBy("version")
)

# Part 2 — OPTIMIZE

**OPTIMIZE** is the compaction command for Delta tables. It does two things:

1. **File compaction** — merges many small Parquet files into fewer larger files (typically targeting ~1 GB per file). This reduces the overhead of opening and reading thousands of tiny files.
2. **Deletion vector resolution** — rewrites the Parquet files that have pending deletion vectors, producing clean files without the deleted rows and removing the `.dv` bitmap files.

### Why you saw OPTIMIZE in `history()` after a DELETE

In the previous notebook you noticed `OPTIMIZE` entries with `auto -> true` immediately after `DELETE` operations. This is **Auto Optimization**, a Databricks feature that runs OPTIMIZE automatically in the background after write-heavy operations. You did not trigger it — Databricks did.

| OPTIMIZE trigger | How | `auto` flag in history |
|------------------|-----|------------------------|
| Manual | `OPTIMIZE table_name` | `false` |
| Auto (background) | Databricks runs after writes/deletes | `true` |

### OPTIMIZE does NOT delete old files

After OPTIMIZE, the old small Parquet files are still on storage — they are just no longer referenced by the current version of the transaction log. They become **orphan files** that `VACUUM` will later remove.

### Run OPTIMIZE Manually

Run `OPTIMIZE` on the table and observe the `operationMetrics` in `history()`.  
Key metrics returned:

| Metric | Meaning |
|--------|---------|
| `numFilesAdded` | New compacted Parquet files written |
| `numFilesRemoved` | Old small files replaced (still on storage until VACUUM) |
| `numDeletionVectorsRemoved` | Bitmap files resolved and removed |
| `filesAdded.avg` | Average size of the new compacted files (bytes) |

In [ ]:
# Run OPTIMIZE manually — compacts files and resolves any pending deletion vectors
spark.sql(f"OPTIMIZE {TABLE}")

dt = DeltaTable.forName(spark, TABLE)

# Confirm DV stats are now cleared
detail = dt.detail().select("numFiles", "sizeInBytes", "statistics").collect()[0]
print("After OPTIMIZE:")
print(f"  numFiles  : {detail['numFiles']}")
print(f"  sizeInBytes: {detail['sizeInBytes']}")
print(f"  statistics: {detail['statistics']}")
# numRowsDeletedByDeletionVectors and numDeletionVectors should now be 0

In [ ]:
# Inspect the OPTIMIZE history entry and its metrics
display(
    dt.history()
    .select("version", "timestamp", "operation", "operationParameters", "operationMetrics")
    .orderBy("version")
)

### OPTIMIZE with Z-ORDER BY

**Z-ORDER BY** is a multi-dimensional clustering technique that co-locates rows with similar values in the same Parquet files.  
When a query filters on the Z-ordered columns, Databricks can skip entire files that cannot contain matching rows — this is called **data skipping**.

```sql
OPTIMIZE table_name ZORDER BY (column1, column2)
```

| Without Z-ORDER | With Z-ORDER |
|-----------------|-------------|
| Rows are laid out in write order | Rows with similar column values are placed in the same files |
| All files may need to be scanned for a filter query | Many files can be skipped entirely based on file-level min/max stats |
| Good for sequential/batch reads | Good for selective point queries and range scans |

> **Z-ORDER tip:** Choose columns that appear frequently in `WHERE` clauses but have **high cardinality** (many distinct values). Low-cardinality columns (like booleans) offer little benefit.

In [ ]:
# OPTIMIZE with Z-ORDER on category and price — useful if queries filter by these columns
spark.sql(f"OPTIMIZE {TABLE} ZORDER BY (category, price)")

dt = DeltaTable.forName(spark, TABLE)

# The history entry will show zOrderBy in operationParameters
display(
    dt.history(3)
    .select("version", "timestamp", "operation", "operationParameters", "operationMetrics")
    .orderBy("version")
)

# Part 3 — VACUUM

**VACUUM** is the garbage collection command for Delta tables. It permanently deletes Parquet files (and deletion vector bitmap files) from storage when:

1. They are **no longer referenced** by the current transaction log (they were replaced by OPTIMIZE or a rewrite), **and**
2. They are **older than the retention threshold** (default: **7 days**).

```
Storage before VACUUM:
  part-000.parquet  (version 0 — original)        ← orphan, 8 days old
  part-001.parquet  (version 0 — original)        ← orphan, 8 days old
  part-002.parquet  (version 3 — OPTIMIZE output) ← referenced by current version
  0001.dv           (deletion vector — resolved)  ← orphan, 2 days old  ← KEPT (< 7d)

Storage after VACUUM (retain 7 days):
  part-002.parquet  ← kept (current)
  0001.dv           ← kept (< 7 days old)
  # part-000, part-001 permanently deleted
```

### The retention threshold and time travel

VACUUM and time travel are in direct tension:

- Time travel requires old Parquet files to remain on storage.
- VACUUM removes old Parquet files to reclaim storage.

The **retention period** (default 7 days, controlled by `delta.deletedFileRetentionDuration`) determines how far back time travel works after VACUUM.

| Scenario | Recommendation |
|----------|-----------------|
| Need 30-day time travel | Set `delta.deletedFileRetentionDuration = interval 30 days` |
| Storage is expensive | Reduce retention to 1–2 days |
| Compliance/audit requirements | Keep default or increase retention |

> **Warning:** Running `VACUUM` with a retention period shorter than 7 days requires `spark.databricks.delta.retentionDurationCheck.enabled = false`. The 7-day floor protects against accidentally deleting files that active transactions still need.

### VACUUM DRY RUN

Before committing to a VACUUM, run `VACUUM ... DRY RUN` to preview which files would be deleted.  
No files are removed — it just returns the list.

In [ ]:
# DRY RUN: see which files are eligible for deletion without actually deleting them
# RETAIN 0 HOURS is used here for demo purposes — in production never go below 168 hours (7 days)
# without disabling the safety check first
dry_run_df = spark.sql(f"VACUUM {TABLE} RETAIN 168 HOURS DRY RUN")
print(f"Files eligible for deletion (retain 7 days): {dry_run_df.count()}")
display(dry_run_df)

### VACUUM — Python API via `DeltaTable.vacuum()`

`DeltaTable.vacuum(retentionHours)` is the Python equivalent of `VACUUM table RETAIN n HOURS`.  
It returns a DataFrame listing the files that were deleted.

In [ ]:
dt = DeltaTable.forName(spark, TABLE)

# vacuum() with default retention (168 hours / 7 days)
# Returns a DataFrame of deleted file paths
deleted_files_df = dt.vacuum(retentionHours=168)

deleted_count = deleted_files_df.count()
print(f"Files permanently deleted by VACUUM: {deleted_count}")
if deleted_count > 0:
    display(deleted_files_df)
else:
    print("No files were old enough to be removed (all files are < 7 days old).")
    print("This is expected in a fresh demo — the files were created just now.")

### What Happens to Time Travel After VACUUM?

Once VACUUM removes the Parquet files for an old version, you can no longer time-travel to that version.  
The transaction log (`_delta_log/`) still records the version, but the underlying data files are gone.

Trying to read a vacuumed version raises an error similar to:
```
AnalysisException: The Delta version X has been vacuumed and is no longer available.
```

Versions whose files are still within the retention window continue to work normally.

In [ ]:
# Time travel is still possible because the files were just created (well within 7-day retention)
df_v0 = spark.sql(f"SELECT * FROM {TABLE} VERSION AS OF 0 ORDER BY product_id")
print(f"Version 0 still readable — row count: {df_v0.count()}")
display(df_v0)

# Part 4 — Full Lifecycle: Delete → OPTIMIZE → VACUUM

The three operations form a natural pipeline:

```
DELETE           OPTIMIZE         VACUUM
────────────     ────────────     ────────────
Marks rows as    Rewrites files   Permanently
deleted in       to remove the    deletes the
deletion vector  deleted rows;    old unreferenced
(no file rewrite)old files become files from
                 orphans          storage
```

The cells below walk through this pipeline step-by-step and read `detail()` after each step to show exactly what changes.

In [ ]:
# Helper: print the key storage metrics from detail()
def print_storage_stats(label: str) -> None:
    dt_ref = DeltaTable.forName(spark, TABLE)
    row = dt_ref.detail().select("numFiles", "sizeInBytes", "statistics").collect()[0]
    stats = row["statistics"] or {}
    print(f"\n{'='*55}")
    print(f"  {label}")
    print(f"{'='*55}")
    print(f"  numFiles                      : {row['numFiles']}")
    print(f"  sizeInBytes                   : {row['sizeInBytes']}")
    print(f"  numDeletionVectors            : {stats.get('numDeletionVectors', 0)}")
    print(f"  numRowsDeletedByDeletionVectors: {stats.get('numRowsDeletedByDeletionVectors', 0)}")

In [ ]:
# Recreate the table fresh for a clean lifecycle demo
spark.createDataFrame(rows).write.format("delta").mode("overwrite").saveAsTable(TABLE)
print_storage_stats("Step 0: Fresh table (12 rows, no deletes)")

In [ ]:
# Step 1: DELETE — deletion vector absorbs the delete, no Parquet rewrite
dt = DeltaTable.forName(spark, TABLE)
dt.delete("in_stock = false")    # removes 3 out-of-stock rows
print_storage_stats("Step 1: After DELETE (in_stock = false — 3 rows removed via DV)")

In [ ]:
# Step 2: OPTIMIZE — resolves deletion vectors, compacts files
spark.sql(f"OPTIMIZE {TABLE}")
print_storage_stats("Step 2: After OPTIMIZE (DV resolved, files compacted)")
# numDeletionVectors and numRowsDeletedByDeletionVectors should both be 0 now
# Old Parquet files + .dv files are now orphans but still on storage

In [ ]:
# Step 3: VACUUM DRY RUN — see what would be removed
dry = spark.sql(f"VACUUM {TABLE} RETAIN 168 HOURS DRY RUN")
print(f"Files eligible for VACUUM (DRY RUN, retain 7 days): {dry.count()}")
# In a live system with files older than 7 days this would list the orphan Parquet + .dv files
# In this fresh demo the files are seconds old — nothing is eligible yet

In [ ]:
# Step 4: VACUUM — permanently remove orphan files (respects 7-day retention)
deleted = dt.vacuum(retentionHours=168)
print(f"Files permanently removed: {deleted.count()}")
print_storage_stats("Step 3: After VACUUM")

# Summary

## Deletion Vectors

| Aspect | Detail |
|--------|--------|
| **What** | A small bitmap (`.dv` file) that marks which row offsets in a Parquet file are deleted |
| **When** | Written by `DELETE` when `delta.enableDeletionVectors = true` (Databricks default) |
| **Benefit** | Avoid expensive Parquet file rewrites for every delete |
| **Cost** | Deleted rows still occupy storage; reads must apply the bitmap filter |
| **Resolved by** | `OPTIMIZE` (rewrites the Parquet file cleanly and drops the `.dv` file) |

## OPTIMIZE

| Aspect | Detail |
|--------|--------|
| **What** | Compacts many small Parquet files into fewer large ones |
| **Also does** | Resolves pending deletion vectors; produces clean files |
| **Trigger** | Manual (`OPTIMIZE table`) or automatic (Databricks background task, `auto -> true` in history) |
| **Z-ORDER BY** | Optional clause to cluster related rows together for better data skipping |
| **Does NOT** | Remove old files from storage — that is VACUUM's job |

## VACUUM

| Aspect | Detail |
|--------|--------|
| **What** | Permanently deletes orphan Parquet and `.dv` files from storage |
| **When eligible** | File is no longer referenced by any transaction log entry AND older than the retention threshold |
| **Default retention** | 168 hours (7 days) — controlled by `delta.deletedFileRetentionDuration` |
| **Impact on time travel** | Vacuumed versions can no longer be read — their data files are gone |
| **DRY RUN** | Preview what would be deleted without actually deleting anything |
| **Python API** | `DeltaTable.vacuum(retentionHours=168)` |

## Pipeline Summary

```
DELETE rows
  └─► Deletion Vector written (fast, no file rewrite)
       └─► OPTIMIZE
             └─► Parquet files rewritten cleanly, DV files removed
                  └─► VACUUM (after retention period)
                        └─► Old Parquet files permanently deleted from storage
```